In [1]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

sys.path.insert(1,os.path.abspath('../..'))
from tools.battle import *

replays = Path("../../data/replays")
replays_1 = Path("../../data/replays/gen9-randombattle")
replays_2 = Path("../../data/replays/gen9-randombattle_2")
replays_3 = Path("../../data/replays/gen9-randombattle_3")

def pull_by_num(num, ns="", parse=False):
    return Battle(f"../../data/replays/gen9-randombattle{ns}/gen9randombattle-{num}.json", parse)

# for testing
bat1 = pull_by_num(2631366401, parse=True)
bat2 = pull_by_num(2631366781, parse=True)
bat3 = pull_by_num(2631368887, parse=True)

In [58]:
# these are the column --names--

# info/columns for the entire battle
colNames = [
    'format','id',
    'p1_win', # {0,1}
    'ratedQ','n_turns',
    'start_time','end_time', 'duration',
    'p1name','p1side','p1elo0','p1elo1',
    'p2name','p2side','p2elo0','p2elo1',
    "type_diversity_diff",
    "num_boosting_abilities_diff",
    "num_move_boosters_diff",
    "total_stat_diff",
    "p1_total_adv",
    "p1_revealed_team_size",
    "p2_revealed_team_size"
]

# info/columns for each pokemon,
# format: M<team_i><mon_j>_<thing>
_cols = [
    'name','speciesId','used',
    'gender','shinyQ','level',
    'ability','item','teraType','role',
    'mv1','mv2','mv3','mv4',
    'type1','type2',
    'hp','atk','def',
    'spa','spd','spe',
    'off'
]

for i in range(2):
    for j in range(6):
        colNames.extend([ f"M{i+1}{j+1}_"+s for s in _cols ])

len(colNames)

299

In [59]:
# these are the column --types--

# info/columns for the entire battle
colTypes = {
    'format': str,
    'id': int,
    'p1_win': int, # {0,1}
    'ratedQ': bool, 
    'n_turns': int,
    'start_time': int,
    'end_time': int,
    'duration': int,
    'p1name':object, 'p1side':int, 'p1elo0':int, 'p1elo1':int,
    'p2name':str, 'p2side':int, 'p2elo0':int, 'p2elo1':int,
    "type_diversity_diff": int,
    "num_boosting_abilities_diff": int,
    "num_move_boosters_diff": int,
    "total_stat_diff": int,
    "p1_total_adv": int,
    "p1_revealed_team_size": int,
    "p2_revealed_team_size": int
}

# info/columns for each pokemon,
# format: M<team_i><mon_j>_<thing>
_cols = [
    'name','speciesId','used',
    'gender','shinyQ','level',
    'ability','item','teraType','role',
    'mv1','mv2','mv3','mv4',
    'type1','type2',
    'hp','atk','def',
    'spa','spd','spe', 'off'
]
_types = [
    str, str, int,
    str, bool, int,
    str, str, str, str,
    str, str, str, str,
    str, str,
    int, int, int,
    int, int, int, int
]

for i in range(2):
    for j in range(6):
        for k, v in zip(_cols, _types):
            colTypes[f"M{i+1}{j+1}_{k}"] = v

len(colTypes)

299

In [68]:
with open('../../data/data_col_types.txt','w') as file:
    file.write(str(col_types))

In [69]:
with open('../../data/data_col_names.txt','w') as file:
    file.write(str(col_names))

In [60]:
# ===========================
def Battle_data(bat) : 
    # `format`, `id#`
    data = [ 
        bat.formatid, 
        int(bat.id.removeprefix(bat.formatid+'-'))
    ]
    
    # `p1_win` {0,1}
    if bat.winner.name == bat.p1.name : 
        data.append(1)
    else : 
        data.append(0)
    
    # `rated`, `n_turns`, `start_time`, `end_time`, `duration`
    data.extend([bat.rated, len(bat.STATES), bat.start_time, bat.end_time, bat.end_time - bat.start_time])
    
    # `p#name`, `p#side`, `p#elo0`, `p#elo1`
    data.extend(vars(bat.p1).values())
    data.extend(vars(bat.p2).values())

    # `type_diversity_diff`, `num_boosting_abilities_diff`, ..., `p1_total_adv` #21
    data.extend(_aux_battle_data(bat)) # [[2]]

    # `p#_revealed_team_size`
    data.append(len(bat.teams[0].keys()))
    data.append(len(bat.teams[1].keys()))
    
    # getting mon info lists (see below for list of entries)
    # -----------------------
    for i in range(2) :
        team=bat.teams_full[i]
        for M in team.keys() :
            usedQ = int(M in bat.teams[i].keys())
            data.extend(mon_info_list(team[M], usedQ)) # [[1]]
    
    return data


# [[1]]
# ===========================
def mon_info_list(Mon, usedQ):
    '''
    Returns list with entries: 
        'name','speciesId','used',
        'gender','shinyQ','level',
        'ability','item','teraType','role',
        'mv1','mv2','mv3','mv4',
        'type1','type2',
        'hp','atk','def',
        'spa','spd','spe'
    '''
    _L = [Mon['name'], Mon['speciesId'], usedQ]
    
    _keys = [
        'gender','shiny','level','ability',
        'item','teraType','role'
        ]
    _L.extend([ Mon[key] for key in _keys ])
    _L.extend(Mon['moves'])

    if len(Mon['types'])==2 : 
        _L.extend(Mon['types'])
    else: 
        _L.extend([Mon['types'][0],"N/A"])
    
    _L.extend(Mon['stats'].values())
    
    return _L


# [[2]]
# ===========================
def _aux_battle_data(battle):
    useful_traits = ["num_move_boosters_diff","num_boosting_abilities_diff"]
    stat_names = ['hp','atk','def','spa','spd','spe']
    red_stat_names = ['hp','off','def','spd','spe'] # reduced stat names where off stands in for max(atk,spa)

    _L = []
    
    # Team construction
    team1 = [FullPokemon(battle.teams_full[0][mon]) for mon in battle.teams_full[0].keys()]
    team2 = [FullPokemon(battle.teams_full[1][mon]) for mon in battle.teams_full[1].keys()]
    teams = [team1,team2]
    
    # `type_diversity_diff`
    p1_types = set(mon.types[i] for mon in team1 for i in range(len(mon.types)))
    p2_types = set(mon.types[i] for mon in team2 for i in range(len(mon.types)))
    _L.append(len(p1_types) - len(p2_types))
    
    # `num_boosting_abilities_diff`
    p1_num_boosting_abilities = sum(int(mon.has_boosting_ability()) for mon in team1)
    p2_num_boosting_abilities = sum(int(mon.has_boosting_ability()) for mon in team2)
    _L.append(p1_num_boosting_abilities - p2_num_boosting_abilities)

    # `num_move_boosters_diff`
    p1_num_move_boosters = sum(int(mon.has_boost_move()) for mon in team1)
    p2_num_move_boosters = sum(int(mon.has_boost_move()) for mon in team2)
    _L.append(p1_num_move_boosters - p2_num_move_boosters)
    
    # `total_stat_diff`
    p1_total_stats = sum(sum(mon.stats[stat] for stat in red_stat_names) for mon in team1)
    p2_total_stats = sum(sum(mon.stats[stat] for stat in red_stat_names) for mon in team2)
    _L.append(p1_total_stats - p2_total_stats)
    
    # `p1_total_adv`
    _L.append(sum(FullPokemon.advantage(team1[m1],team2[m2]) for m1 in range(6) for m2 in range(6)))

    return _L

In [64]:
# ===========================
DATA = []
errs = []

for file in replays.rglob("*.json") : 
    try : 
        bat = Battle(file, parse=True)
        DATA.append(Battle_data(bat))
    except : 
        # print(f"error with {file.stem.removeprefix("gen9randombattle-")}")
        errs.append(file.stem.removeprefix("gen9randombattle-"))

len(errs)

0

In [65]:
df = pd.DataFrame(DATA, columns=col_names)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12778 entries, 0 to 12777
Columns: 299 entries, format to M26_off
dtypes: bool(2), float64(8), int64(53), object(236)
memory usage: 29.0+ MB


In [ ]:
df=df.astype(col_types)
df.info()

In [66]:
with open('../../data/data_cleaned.csv','w') as file:
    file.write(df.to_csv(index=False))

In [ ]:
df = pd.read_csv("../../data/data_test.csv", low_memory=False)
df.info()